# 线上漂移监控

**面试回答：**监控特征、缺失率、输出和延迟标签质量；PSI 提示分布变化，不等于模型必然失效，告警必须有负责人和回退动作。

## 真实案例

促销前后订单金额分布变化，比较训练窗口与本周窗口的 PSI。

In [1]:
import numpy as np  # 导入 NumPy 手写 PSI。
train=np.array([20,25,30,35,40,45,50,55,60,65,70,75],dtype=float)  # 构造训练期订单金额。
current=np.array([25,35,45,55,65,75,85,95,110,130,150,180],dtype=float)  # 构造促销期金额。
print('训练金额=',train.tolist())  # 输出训练分布样本。
print('本周金额=',current.tolist())  # 输出当前分布样本。

训练金额= [20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0]
本周金额= [25.0, 35.0, 45.0, 55.0, 65.0, 75.0, 85.0, 95.0, 110.0, 130.0, 150.0, 180.0]


## Baseline / 基线

基线只看 QPS 正常，无法发现特征分布已经变化。

In [2]:
qps_train=1000  # 设定训练期服务吞吐量。
qps_current=1020  # 设定本周服务吞吐量。
print('QPS变化率=',round((qps_current-qps_train)/qps_train,3))  # 输出表面稳定的服务指标。
print('仅QPS正常不能说明模型输入稳定。')  # 说明基线不足。

QPS变化率= 0.02
仅QPS正常不能说明模型输入稳定。


In [3]:
edges=np.array([0,35,50,65,80,200],dtype=float)  # 定义固定监控分桶。
train_count=np.histogram(train,bins=edges)[0]  # 统计训练期每桶数量。
current_count=np.histogram(current,bins=edges)[0]  # 统计本周每桶数量。
train_rate=(train_count+.5)/(train_count.sum()+.5*len(train_count))  # 用平滑计算训练比例。
current_rate=(current_count+.5)/(current_count.sum()+.5*len(current_count))  # 用平滑计算当前比例。
psi=float(np.sum((current_rate-train_rate)*np.log(current_rate/train_rate)))  # 手写 PSI 公式。
print('分桶/训练率/当前率=',list(zip(edges[:-1].tolist(),np.round(train_rate,3).tolist(),np.round(current_rate,3).tolist())))  # 输出 PSI 中间量。
print('PSI=',round(psi,3))  # 输出漂移指标。

分桶/训练率/当前率= [(0.0, 0.241, 0.103), (35.0, 0.241, 0.172), (50.0, 0.241, 0.103), (65.0, 0.241, 0.172), (80.0, 0.034, 0.448)]
PSI= 1.342


## 结果解读

PSI 高说明金额分布与训练不同，可能来自促销、埋点变化或人群变化；需结合缺失率、输出分数和延迟标签质量判断。

In [4]:
action='人工复核并启动回放' if psi>.2 else '继续监控'  # 将告警阈值连接到明确动作。
print('告警动作=',action)  # 输出运行手册动作。
print('缺失率/模型版本/特征版本也应与PSI一起入账。')  # 输出关联监控项。
print('教学分桶固定，线上应按特征语义和季节性校准阈值。')  # 说明边界。

告警动作= 人工复核并启动回放
缺失率/模型版本/特征版本也应与PSI一起入账。
教学分桶固定，线上应按特征语义和季节性校准阈值。


## 失败案例与修复

若 PSI 告警后直接自动再训练，可能把脏埋点或标签延迟写入模型；修复是先检查数据合同、回放和可回退版本。

In [5]:
auto_retrain=psi>.2  # 构造只凭 PSI 自动训练的失败规则。
print('失败自动再训练=',auto_retrain)  # 输出危险动作。
print('修复顺序：检查特征合同→延迟标签回放→shadow验证→canary或回退。')  # 输出安全流程。
print('漂移不等于退化，质量指标到达前不能声称模型失效。')  # 说明因果边界。
print('每个告警需要负责人、窗口、阈值和处置记录。')  # 输出运行要求。

失败自动再训练= True
修复顺序：检查特征合同→延迟标签回放→shadow验证→canary或回退。
漂移不等于退化，质量指标到达前不能声称模型失效。
每个告警需要负责人、窗口、阈值和处置记录。


In [6]:
assert len(train)>=5  # 保护训练样本数。
assert psi>0  # 保护检测到分布差异。
assert qps_current>qps_train  # 保护 QPS 表面正常反例。
assert action=='人工复核并启动回放'  # 保护告警动作。